In [1]:
import pandas as pd
import geopandas as gpd
import win32com.client as com
import os

c:\Users\Roberto Ponce López\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
FOLDER = r'C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara'
RED_FOLDER = r'Red Base GDL\RedBase Final\version base para rezonificacion'

In [3]:
current_network = os.path.join(FOLDER, RED_FOLDER, "RedFinal - rezonificacion - v2.ver")
Visum = com.Dispatch("Visum.Visum")
Visum.LoadVersion(current_network)
C = com.constants

In [6]:
links = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "FromNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "Name": [i[1] for i in Visum.Net.Links.GetMultiAttValues("Name")],
    "TSysSet": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TSysSet")],
    "TypeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TypeNo")],
    "FINAL_FILTER": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FINAL_FILTER")],
    "HAS_AFORO": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HAS_AFORO")],
})
print(f"Total of: {len(links):,}")

Total of: 506,860


In [7]:
links_with_aforo = links[links["HAS_AFORO"]==1].copy()
links_with_aforo["FINAL_FILTER"].value_counts()

FINAL_FILTER
1.0    640
Name: count, dtype: int64

In [8]:
# Limpiar espacios para comparar nombres
nombres = links["Name"].astype("string").str.strip()

# Nombres válidos de calles con aforo
nombres_con_aforo = nombres[
    links["HAS_AFORO"].eq(True)
    & nombres.notna()
    & nombres.ne("")
]

# Actualizar todos los links que comparten esos nombres
mask = nombres.isin(nombres_con_aforo)
print("VALUE COUNTS of Links that have the same street name as a link with AFORO registered")
print(links.loc[mask]["FINAL_FILTER"].value_counts())
print()

print("Make sure they all have FINAL_FILTER==1 , so they don't close for PrT")
links.loc[mask, "FINAL_FILTER"] = 1 #Para que no se vayan a cerrar por ser locales al tener "FINAL_FILTER==0"
print(links.loc[mask]["FINAL_FILTER"].value_counts())


VALUE COUNTS of Links that have the same street name as a link with AFORO registered
FINAL_FILTER
1.0    21106
0.0     1258
Name: count, dtype: int64

Make sure they all have FINAL_FILTER==1 , so they don't close for PrT
FINAL_FILTER
1.0    22364
Name: count, dtype: int64


In [9]:
# Make sure all links with aforo registered on themselves, are FINAL_FILTER==1
print(links.loc[links["HAS_AFORO"].eq(True)]["FINAL_FILTER"].value_counts())

links.loc[links["HAS_AFORO"].eq(True), "FINAL_FILTER"] = 1
print(links.loc[links["HAS_AFORO"].eq(True)]["FINAL_FILTER"].value_counts())

FINAL_FILTER
1.0    640
Name: count, dtype: int64
FINAL_FILTER
1.0    640
Name: count, dtype: int64


In [10]:
#Reimport FINAL_FILTER column in Visum
links_visum = Visum.Net.Links

links = links.reset_index(drop=True)
links.index = links.index + 1

final_filter = list(zip(links.index, links["FINAL_FILTER"].astype(int)))

links_visum.SetMultiAttValues("FINAL_FILTER", final_filter)